# Stage 7 · Cell-type annotation

Aggregate all L1 + L2 labels into the master metadata (`5kCG100k3C_summary`), assign subtype names, and write the pseudobulk file-lists used downstream.

These are the original pipeline notebooks, concatenated in execution order with paths normalized to `ENTEX_ROOT`. They document the method and run per tissue / major type across the full raw dataset (heavy compute); they are shown for reference and are **not re-executed in the book**. Example group shown where templated.

In [1]:
# === Reproduction setup ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT); import repro_guard


## 7 · master annotation & metadata

_Source: `clustering/merged/summary.ipynb`_

In [2]:
import numpy as np
import pandas as pd
from glob import glob
from gliderport.preset import notebook_snakemake


In [3]:
indir = f'{ENTEX_ROOT}/clustering/merged/'
group_list = [xx.split('/')[-2] for xx in glob(f'{indir}L2_new/*/5kCG_embed.h5ad')]
print(len(group_list))

In [4]:
notebook_snakemake(
    work_dir=f'{indir}L2_new/',
    notebook_dir=f'{indir}template/',
    groups=group_list,
    default_cpu=16,
    default_mem_gb=64,
    redo_prepare=True,
)

In [5]:
import joblib
import anndata
import scanpy as sc

In [6]:
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [7]:
L1_list = np.sort([xx.split('/')[-2] for xx in glob(f'{indir}L2/*/5kCG100k3C_embed.h5ad')])


In [8]:
ct = 'Epi-AdrCtx'
adata = anndata.read_h5ad(f'{indir}L2/{ct}/5kCG100k3C_embed.h5ad')
adata.obs.index = adata.obs.index.astype(str)
cc = joblib.load(f'{indir}L2/{ct}/ConcensusClustering.model.lib')


In [9]:
for ct in L1_list:
    adata = anndata.read_h5ad(f'{indir}L2/{ct}/5kCG100k3C_embed.h5ad')
    cc = joblib.load(f'{indir}L2/{ct}/ConcensusClustering.model.lib')
    adata.obs['leiden_init'] = cc.step_data[1][0]
    adata.obs['leiden_init'] = 'c' + adata.obs['leiden_init'].astype(str)
    count_init = len(adata.obs['leiden_init'].unique())
    count_cons = len(adata.obs['leiden_cons'].unique())
    if count_init!=count_cons:
        print(ct)
        cc.target_accuracy = 0
        cc.supervise_learning()
        cc.final_evaluation()
        adata.obs['leiden_init'] = cc.label.copy()
    adata.write_h5ad(f'{indir}L2/{ct}/5kCG100k3C_embed.h5ad')


In [10]:
# tissue_list = np.sort([xx.split('/')[-2] for xx in glob(f'{indir}../../tissue/*/')])
# tissue_list
tissue_list = np.sort(['AG', 'B', 'EG', 'LG', 'LV', 'M1C', 'NTb', 'P', 'PBMC', 'PI',
       'RAA', 'SB', 'ST', 'Sk', 'SkGcn', 'TrCo'])
tissue_color = {xx:yy for xx,yy in zip(tissue_list, sns.color_palette('tab20', len(tissue_list)))}


In [11]:
result = pd.DataFrame(result, columns=['L1', 'CG_npc', '3C_npc']).set_index('L1')

In [12]:
result.sort_values('3C_npc')

In [13]:
result.sort_values('3C_npc')

In [14]:
result.sort_values('CG_npc')

In [15]:
# result = []
# for ct in L1_list:
#     adata_mc = anndata.read_h5ad(f'{indir}L2/{ct}/5kCG_embed.h5ad')
#     adata_3c = anndata.read_h5ad(f'{indir}L2/{ct}/100k3C_embed.h5ad')
#     adata_3c = adata_3c[adata_mc.obs.index].copy()
#     adata = anndata.read_h5ad(f'{indir}L2/{ct}/5kCG100k3C_embed.h5ad')
#     npc_cg = [int(xx.split('_')[1][1:]) for xx in adata_mc.obsm.keys() if '_seuratL2_tsne' in xx][0]
#     # npc_3c = [int(xx.split('_')[1][1:]) for xx in adata_3c.obsm.keys() if '100k3C_u' in xx][0]
#     npc_3c = [int(xx.split('_')[1][2:]) for xx in adata_3c.obsm.keys() if '_seuratL2_tsne' in xx][0]
#     result.append([ct, npc_cg, npc_3c])


In [16]:
import warnings
warnings.filterwarnings('ignore')

In [17]:
npc = pd.read_csv(f'{indir}npc.tsv', sep='\t', header=None, index_col=0, names=['npc_cg', 'npc_3c'])
npc

In [18]:
# coord_base = 'tsne'
# for ct in L1_list:
#     adata = anndata.read_h5ad(f'{indir}L2/{ct}/5kCG100k3C_embed.h5ad')
#     count_init = len(adata.obs['leiden_init'].unique())
#     count_cons = len(adata.obs['leiden_cons'].unique())
#     ds = 150/np.sqrt(adata.shape[0])
#     fig, axes = plt.subplots(1, 3, figsize=(12,3), dpi=300)
#     fig.suptitle(f'{ct} {adata.shape[0]} {count_init} {count_cons}', fontsize=15)
#     ax = axes[0]
#     _ = categorical_scatter(data=adata.obs,
#                             ax=ax,
#                             coord_base=coord_base,
#                             hue='ClusterTissue',
#                             # text_anno='ClusterTissue',
#                             s=ds,
#                             labelsize=12,
#                             max_points=None,
#                             palette='tab20',
#                             scatter_kws={'rasterized':True},
#                             legend_kws={'fontsize':6},
#                             show_legend=True,
#                            )
    
#     ax = axes[1]
#     _ = categorical_scatter(data=adata.obs,
#                             ax=ax,
#                             coord_base=coord_base,
#                             hue='leiden_init',
#                             text_anno='leiden_init',
#                             s=ds,
#                             labelsize=12,
#                             max_points=None,
#                             palette='tab20',
#                             scatter_kws={'rasterized':True},
#                             # show_legend=True
#                            )
    
#     ax = axes[2]
#     _ = categorical_scatter(data=adata.obs,
#                             ax=ax,
#                             coord_base=coord_base,
#                             hue='Tissue',
#                             # text_anno=xx,
#                             s=ds,
#                             labelsize=8,
#                             max_points=None,
#                             palette=tissue_color,
#                             scatter_kws={'rasterized':True},
#                             show_legend=True
#                            )
#     plt.tight_layout()

        

In [19]:
adata_list = glob(f'{indir}L2/*/5kCG100k3C_embed.h5ad')
adata = []
for file in adata_list:
    tmp = anndata.read_h5ad(file)
    adata.append(tmp)
    
adata = anndata.AnnData.concatenate(*adata, index_unique=None).copy()
adata

In [20]:
tmp = anndata.read_h5ad(f'{indir}L1/5kCG100k3C_embed.h5ad')
adata.obs['L1_annot'] = tmp.obs['L1'].copy()
adata.obsm['X_tsne'] = tmp.obsm['X_tsne'].copy()


In [21]:
L1map = 'c' + adata.obs['L1_annot'].value_counts().reset_index().reset_index().set_index('L1_annot')['index'].astype(str)
adata.obs['L1'] = adata.obs['L1_annot'].map(L1map)

In [22]:
adata.obs['L1'].astype(str) + '|' + adata.obs['leiden_init'].astype(str)

In [23]:
adata.write_h5ad('5kCG100k3C_summary.h5ad')

In [24]:
count = (adata.obs['L1'].astype(str) + '|' + adata.obs['leiden_init'].astype(str)).value_counts()
print(count.shape[0])

In [25]:
count.loc[count>100]

In [26]:
adata.obs[['L1_annot', 'leiden_init']].astype(str).drop_duplicates()['L1_annot'].value_counts()

In [27]:
adata.obs[['L1_annot', 'leiden_cons']].astype(str).drop_duplicates()['L1_annot'].value_counts()

In [28]:
adata = anndata.read_h5ad('5kCG100k3C_summary.h5ad')
adata

In [29]:
adata.obs['allcpath'] = '/gale/netapp/entex/ENTEx/allc_CGN/' + adata.obs.index + '.CGN-Both.allc.tsv.gz'
adata.obs['coolpath'] = 'gs://ecker-zhoujt-analysis/ENTEx/tissue/' + adata.obs['Tissue'].astype(str) + '/impute/10K/' + adata.obs.index + '.cool'


In [30]:
adata.obs['cluster'] = adata.obs['L1'].astype(str) + '-' + adata.obs['leiden_init'].astype(str)


In [31]:
adata.obs.astype(str).sort_values(by=['L1', 'leiden_init'])[['allcpath', 'cluster']].to_csv('allclist_cluster.csv', index=False, header=False)
adata.obs.astype(str).sort_values(by=['L1', 'leiden_init'])[['coolpath', 'cluster']].to_csv('coollist_cluster.csv', index=True, header=False)


In [32]:
cluster_meta = adata.obs[['L1', 'leiden_init', 'L1_annot']].drop_duplicates()
cluster_meta.columns = ['L1', 'L2', 'L1_annot']
cluster_meta.index = cluster_meta['L1'].astype(str) + '-' + cluster_meta['L2'].astype(str)


In [33]:
cluster_meta.sort_values(by=['L1_annot', 'L2']).to_csv('cluster_meta.tsv', sep='\t', header=True, index=True)

In [34]:
mcds_path_list = np.sort(glob(f'{ENTEX_ROOT}/mcds/*mcds'))
print(len(mcds_path_list))

In [35]:
from ALLCools.mcds import MCDS
mcds = MCDS.open(mcds_path_list, var_dim='gene')
mcds

In [36]:
selc = mcds.get_index('cell').intersection(adata.obs.index)
print(selc.shape[0])

In [37]:
mcds = mcds.sel({'cell':selc, 'mc_type':'CGN'})

In [38]:
mcds = mcds.remove_chromosome(var_dim='gene', exclude_chromosome=['chrX', 'chrY', 'chrM', 'chrL'])


In [39]:
mcds['cluster'] = adata.obs['L1'].astype(str) + '|' + adata.obs['leiden_init'].astype(str)

In [40]:
bulk_mcds = mcds.groupby('cluster').sum(dim='cell')

In [41]:
mc = bulk_mcds.sel({'count_type':'mc'})['gene_da'].to_pandas()
cov = bulk_mcds.sel({'count_type':'cov'})['gene_da'].to_pandas()

In [42]:
from ALLCools.mcds.utilities import calculate_posterior_mc_frac

ratio = calculate_posterior_mc_frac(mc.values, cov.values)
ratio = pd.DataFrame(ratio, index=mc.index, columns=mc.columns)


In [43]:
ratio.to_hdf('leideninit_gene_mCG.hdf', key='data')

In [44]:
gene3c_path_list = glob('/home/jzhou_salk_edu/sky_workdir/entex/gene3c/*hdf')
print(len(gene3c_path_list))


In [45]:
gene3c = []
for gene3c_path in gene3c_path_list:
    gene3c.append(pd.read_hdf(gene3c_path))
    
gene3c = pd.concat(gene3c, axis=0)
    

In [46]:
gene3c['cluster'] = adata.obs['L1'].astype(str) + '|' + adata.obs['leiden_init'].astype(str)
bulk_gene3c = gene3c.groupby('cluster').mean()
print(len(ratio.columns.intersection(bulk_gene3c.columns)))


In [47]:
bulk_gene3c = bulk_gene3c[ratio.columns]
bulk_gene3c.columns.name = 'gene'
bulk_gene3c.to_hdf('leideninit_gene_3C.hdf', key='data')


In [48]:
adata.obs